# Eksperiment 2-9 - Razlika formi SMA(7)

In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from experiment_utils import run_experiment

import warnings
warnings.filterwarnings('ignore')

## Podaci

In [2]:
df = pd.read_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['date'] = pd.to_datetime(df['date'], format='mixed')
df = df.sort_values('date').reset_index(drop=True)

print(f'Ukupno meceva: {len(df)}')
print(df['result'].value_counts())

Ukupno meceva: 1239
result
 1    582
-1    371
 0    286
Name: count, dtype: int64


## SMA(7) i razlika formi

In [3]:
home_rows = df[['date', 'home_team', 'result']].rename(columns={'home_team': 'team', 'result': 'team_result'})
away_rows = df[['date', 'away_team', 'result']].copy()
away_rows['team_result'] = -away_rows['result']
away_rows = away_rows[['date', 'away_team', 'team_result']].rename(columns={'away_team': 'team'})

team_results = pd.concat([home_rows, away_rows]).sort_values('date').reset_index(drop=True)
team_results['sma7'] = (
    team_results.groupby('team')['team_result']
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

df = df.merge(team_results.rename(columns={'team': 'home_team', 'sma7': 'home_form_sma7'})[['date', 'home_team', 'home_form_sma7']], on=['date', 'home_team'], how='left')
df = df.merge(team_results.rename(columns={'team': 'away_team', 'sma7': 'away_form_sma7'})[['date', 'away_team', 'away_form_sma7']], on=['date', 'away_team'], how='left')
df = df.dropna(subset=['home_form_sma7', 'away_form_sma7']).reset_index(drop=True)

df['form_diff_sma7'] = df['home_form_sma7'] - df['away_form_sma7']
print(f'Meceva nakon filtriranja: {len(df)}')

Meceva nakon filtriranja: 431


## Priprema i pokretanje

In [4]:
X = df[['form_diff_sma7']].values
le = LabelEncoder()
y = le.fit_transform(df['result'].values)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_t = torch.tensor(y_train,    dtype=torch.long)
X_test_t  = torch.tensor(X_test_sc,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,     dtype=torch.long)

run_experiment(X_train_t, y_train_t, X_test_t, y_test_t)

Run 1/10
Accuracy: 0.5402

Run 2/10
Accuracy: 0.5172

Run 3/10
Accuracy: 0.5402

Run 4/10
Accuracy: 0.5402

Run 5/10
Accuracy: 0.5402

Run 6/10
Accuracy: 0.5402

Run 7/10
Accuracy: 0.5172

Run 8/10
Accuracy: 0.5172

Run 9/10
Accuracy: 0.5172

Run 10/10
Accuracy: 0.5402

Mean accuracy: 0.5310
Classification report for last run:
              precision    recall  f1-score   support

    Away Win       0.46      0.57      0.51        21
        Draw       0.00      0.00      0.00        23
    Home Win       0.57      0.81      0.67        43

    accuracy                           0.54        87
   macro avg       0.35      0.46      0.39        87
weighted avg       0.39      0.54      0.46        87

Confusion matrix for last run:
               Pred Away Win  Pred Draw  Pred Home Win
True Away Win             12          0              9
True Draw                  6          0             17
True Home Win              8          0             35
